# 7 · Boolean search

**Owner:** Member 1 · **Branch:** `dev-boolean` · **Code:** `src/discovery/index/boolean.py` → class `BooleanSearch` · **Writes to:** `eval/results/boolean.json`

**What it means:** A repository either matches or it doesn't. The inverted index lists, for every word, the repositories that contain it. **AND** keeps repositories in both lists, **OR** in either, **NOT** removes a list from all repositories.

**Key idea:** It is the floor every ranked method has to beat, and its inverted index is the base the others reuse. It has no notion of "more relevant": a match is a match. To still return a top 10, hits are ordered by how many query words they contain.

Full guide with starter code, and the Results sheet: [team plan](https://claude.ai/artifact/Rst8GHxAbeHavg4JS2RZV3) (tab 7 Boolean search). Handoff and setup: `docs/handoff-methods.md`.

### Settings to try
| Setting | Values | Why |
|---|---|---|
| default operator | AND, then OR if AND finds nothing | How a free-text query becomes a Boolean one. |
| fields | name, description, topics, README | Which text goes into the index. |
| ranking of hits | number of query words matched, then corpus order | Boolean has no score; this is only a tie-break. |

**Fixed:** tokens from the shared tokenize(); doc_id = full_name; score = number of query words the repository contains

### What we agreed (same for all five)
- **One tokenizer:** `tokenize()` from `discovery.preprocess.text`, for documents *and* queries. Never a private one.
- **One interface:** your class extends `SearchMethod` (`src/discovery/rank/base.py`): `fit(corpus)` → self, `search(query, k)` → `[{"doc_id": full_name, "score": float}]`, best first. `check_hits()` checks it.
- **Tune on the 5 development queries only** (`DEV`, replaced by the real ones in Phase 13). The 20 test queries are for the final number only.
- **Seed 42**, paths from `PATHS`, and don't edit the shared files (`rank/base.py`, `rank/devkit.py`, `preprocess/text.py`, `config/config.yaml`) on your branch: ask Member 6.
- Commit this notebook **with its outputs** on `dev-boolean`. No data files in git.

## 1. Setup: load the corpus and the shared tokenizer

**Check:** it prints 8,223 repositories, and `node.js` and `c#` come out of `tokenize()` in one piece.

In [ ]:
%load_ext autoreload
%autoreload 2

import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from discovery.config import PATHS, SEED, set_seed
from discovery.preprocess.text import doc_text, tokenize      # the ONE shared tokenizer (Phase 6)
from discovery.rank.base import check_hits
from discovery.rank.devkit import DEV, load_corpus, measure, save_results, show_top
from discovery.index.boolean import BooleanSearch

set_seed()
corpus = load_corpus()                                   # data/processed/corpus.parquet
ids = corpus["full_name"].tolist()                       # doc_id = full_name, "owner/name"
text = doc_text(corpus).tolist()                         # name + description + topics + README
print(f"{len(corpus):,} repositories |", tokenize("Built with Node.js and C#"))
print("development queries:", *DEV, sep="\n  ")

## 2. Build: Boolean search

Each repository's text goes through the shared `tokenize()` once; every distinct term adds the repository's row number to that term's posting list.

**Check:** it runs once on the whole corpus without errors. Write down how long it took: it goes in the sheet.

<details><summary>Starter code from the team plan (open it if you are stuck; adapt it, don't just paste)</summary>

```python
from collections import defaultdict

postings = defaultdict(set)                     # term -> row numbers of the repositories that contain it
for i, t in enumerate(text):
    for term in set(tokenize(t)):
        postings[term].add(i)
ALL = set(range(len(ids)))
print(f"{len(postings):,} terms | 'inventory' in {len(postings['inventory']):,} repositories")
```
</details>

## 3. Search: a free-text query in, a ranked list out

A small parser for AND, OR, NOT and brackets. Each word is looked up in the index; the operators are set operations on the posting lists.

**Check:** the top 10 for `DEV[0]` look plausible when you read their names and descriptions.

<details><summary>Starter code from the team plan (open it if you are stuck; adapt it, don't just paste)</summary>

```python
import re

def boolean(query: str) -> set[int]:
    """'inventory AND (nestjs OR express) AND NOT tutorial'. NOT binds tightest, then AND, then OR.
    Two words next to each other mean AND."""
    toks = re.findall(r"\(|\)|\bAND\b|\bOR\b|\bNOT\b|[^\s()]+", query)
    def expr():
        r = term()
        while toks and toks[0] == "OR":
            toks.pop(0); r = r | term()
        return r
    def term():
        r = factor()
        while toks and toks[0] not in ("OR", ")"):
            if toks[0] == "AND": toks.pop(0)
            r = r & factor()
        return r
    def factor():
        t = toks.pop(0)
        if t == "NOT": return ALL - factor()
        if t == "(":
            r = expr(); toks.pop(0); return r            # drop the ")"
        words = tokenize(t)
        return set.intersection(*(postings.get(w, set()) for w in words)) if words else ALL
    return expr()

print(len(boolean("inventory AND nestjs")), len(boolean("inventory AND (nestjs OR express)")))
```
</details>

## 4. Move it into `src/discovery/index/boolean.py` as `BooleanSearch`

The stub is already there with the right signature. Fill in `fit` and `search` (and `params`), then load it here with `discovery.index.boolean.BooleanSearch`: autoreload picks up your edits.

**Check:** `check_hits` passes below, and in a terminal `pytest tests/test_methods.py -k boolean` passes (it was skipped until now).

<details><summary>The search method, as in the plan (open it if you are stuck; adapt it, don't just paste)</summary>

```python
    def search(self, query, k=10):
        terms = tokenize(query)
        hits = set.intersection(*(self.postings.get(t, set()) for t in terms)) if terms else set()
        if not hits:                                      # AND found nothing: fall back to OR
            hits = set().union(*(self.postings.get(t, set()) for t in terms))
        score = {i: sum(i in self.postings.get(t, ()) for t in terms) for i in hits}
        best = sorted(hits, key=lambda i: (-score[i], i))[:k]
        return [{"doc_id": self.ids[i], "score": float(score[i])} for i in best]
```
</details>

In [ ]:
method = BooleanSearch().fit(corpus)
check_hits(method.search(DEV[0], 10), k=10)
method.params()

## 5. Test exact matching: the set rules must hold

Boolean retrieval is exact, so it can be checked exactly. These identities hold for any two words; if one fails, the parser or the index is wrong.

**Check:** All asserts pass, and the 5 repositories you opened really contain both words (search the README on GitHub with Ctrl+F).

<details><summary>The check, as in the plan (open it if you are stuck; adapt it, don't just paste)</summary>

```python
A, B = boolean("inventory"), boolean("nestjs")
assert boolean("inventory AND nestjs") == A & B
assert len(boolean("inventory OR nestjs")) == len(A) + len(B) - len(A & B)
assert len(boolean("NOT inventory")) + len(A) == len(ids)
assert boolean("inventory nestjs") == boolean("inventory AND nestjs")      # implicit AND
assert boolean("NOT inventory AND nestjs") == (ALL - A) & B                # NOT binds tighter than AND
# open 5 hits of "inventory AND nestjs" on GitHub: does each really contain both words?
print(corpus.loc[sorted(A & B)[:5], ["full_name", "description"]])
```
</details>

## 6. Read your top 10 on the development queries

Read them. Write down two good results and two bad ones, and why each bad one ranked high (in the markdown cell under the output).

**Check:** you can say in one sentence why each bad result ranked high.

In [ ]:
show_top(method, corpus, DEV)

*Two good results, two bad results, and why:*

- 
- 
- 
- 

## 7. Tune on the 5 development queries only

There is little to tune. Compare AND-only vs AND-then-OR on the development queries: how many queries return nothing with AND only?

Until Phase 13 gives the development labels, compare settings by reading the top 10 (step 6). When `eval/qrels.csv` has the development labels, use `discovery.evaluation.metrics.ndcg_at_k` and keep the setting with the best mean NDCG@10. **Never tune on the 20 test queries.**

**Check:** a small table: each setting tried, what changed in the top 10 (later: mean NDCG@10). Keep the best and note it in the sheet.

## 8. Speed, results file, hand in

`measure()` times the build and the queries; `save_results()` writes `eval/results/boolean.json` in the format the Results sheet imports. The metrics stay empty until Phase 13–14: then Member 6's `evaluate_method(method, split="test")` fills them in and you import the file in the **Results sheet** tab of the team plan, add your observations and download the PDF.

**Check:** `eval/results/boolean.json` exists; this notebook is committed with its outputs on `dev-boolean`; `pytest` passes.

In [ ]:
speed = measure(lambda: BooleanSearch().fit(corpus), DEV)
print(speed)
save_results(method, speed)

## Extra (optional): Phrase queries with a positional index

Store word positions as well (`term → {row: [positions]}`) and support `"point of sale"` as a phrase: the words must appear next to each other. Compare precision@10 on queries that contain a phrase.

## What we found

*(fill in at the end: build time, the best setting, two good and two bad results, and anything surprising)*